**Purpose:** `metrics.ipynb` — part of `03_portfolio/LSTM_returns/_v522_returns_prediction` (see the stage README).

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


# LSTM Returns Prediction — Results Summary & Discussion

## 1. Model Overview

The model is a **multi-output LSTM** designed to forecast next-day returns for 11 U.S. sector ETFs (XLB, XLC, XLE, XLF, XLI, XLK, XLP, XLRE, XLU, XLV, XLY). Rather than training one model per asset, a **shared per-asset LSTM encoder** learns temporal dynamics jointly across all sectors, followed by a **cross-asset Transformer mixer** that captures cross-sectional interactions. A joint MLP fusion head then produces per-asset return forecasts.

The architecture directly motivates a multi-task framing: all sectors share common macroeconomic drivers (interest rates, VIX, commodity cycles), so a shared encoder benefits from parameter sharing, while the Transformer mixer allows the model to explicitly reason about cross-sector relationships at each time step. Sector-specific information enters implicitly through distinct input time series for each asset.

---

## 3. Training Setup

- **Loss:** Masked MSE or masked Huber (Optuna selects between them). Masking ensures assets with missing data on a given day are excluded from gradient updates.
- **Optimizer:** AdamW with optional cosine or plateau LR scheduling and gradient clipping.
- **Early stopping:** based on validation loss (not IC), with a held-out last year of the training window used as the early-stop set.
- **HPO:** Bayesian search via Optuna (TPE sampler) over 3 expanding walk-forward folds × 3 seeds per trial; the objective is **mean Spearman IC** across folds and seeds.
- **Ensemble:** The top-N trials (by validation IC) are refitted on the full pre-test window; their predictions are averaged. The mean prediction feeds Black–Litterman as views, and the variance across models provides uncertainty estimates for the confidence parameter.

---

## 4. Evaluation Metrics

Results are reported across four **sentiment feature sources** (`TECH`, `NEWS`, `REDDIT`, `REDDIT+NEWS`) and four **model variants**:

| Variant | Description |
|---------|-------------|
| `ensemble` | Average of top-N trial predictions |
| `rank0` | Best single trial (by validation IC) |
| `rank1` | Second-best single trial |
| `rank2` | Third-best trial |

Three error metrics are used:
- **MSE / RMSE** — magnitude of prediction error (absolute, in return space)
- **MAPE** — mean absolute percentage error; interpretable but sensitive to near-zero returns
- **IC (Pearson)** and **Rank IC (Spearman)** — cross-sectional correlation between predicted and realized returns at each time step; IC measures directional ranking ability rather than absolute accuracy

---

## 5. Results Discussion

### 5.1 Aggregate Error Metrics

The table below summarizes aggregate RMSE and MAPE for the ensemble variant across sources:

| Source | RMSE (mean) | MAPE (mean) | IC Mean | Rank IC Mean |
|--------|-------------|-------------|---------|--------------|
| TECH | 0.01135 | 1.2157 | 0.0256 | 0.0146 |
| NEWS | 0.01131 | 1.2933 | 0.0094 | 0.0141 |
| REDDIT | 0.01139 | 1.5289 | 0.0087 | 0.0088 |
| REDDIT+NEWS | 0.01140 | 1.4940 | 0.0196 | 0.0339 |

**TECH features yield the best ensemble MAPE (1.22%) and the highest IC mean (0.026)**, suggesting that technical indicators alone provide the most consistent signal for next-day return prediction in absolute magnitude terms. NEWS-only follows closely on RMSE but with higher MAPE and lower IC. REDDIT alone performs worst in MAPE (1.53%), consistent with noisier, less structured sentiment data.

Interestingly, **REDDIT+NEWS achieves the highest Rank IC mean (0.034)**, which suggests that combining both text sources improves cross-sectional rank ordering of assets, even if absolute errors are slightly higher. This is the metric most relevant for portfolio construction, where getting the relative ranking of assets right matters more than point accuracy.

### 5.2 Ensemble vs. Individual Trials

Across all sources, the **ensemble consistently outperforms individual ranked trials** on MAPE, confirming the value of the top-N averaging approach:

- For NEWS: ensemble MAPE = 1.29 vs. rank0 = 1.19, rank1 = 1.77, rank2 = 1.57
- For TECH: ensemble MAPE = 1.22 vs. rank0 = 1.28, rank1 = 1.29, rank2 = 1.47
- For REDDIT: ensemble MAPE = 1.53 vs. rank0 = 2.49 (notably poor), rank2 = 1.22

One exception is noteworthy: `rank0` for NEWS shows lower MAPE than the ensemble (1.19 vs. 1.29), and `rank2` for REDDIT outperforms the ensemble (1.22 vs. 1.53). This suggests that **averaging over weaker trials can occasionally hurt** if the lower-ranked models introduce noise — particularly for noisier sources like Reddit. A stricter top-N cutoff or weighted ensemble could help here.

The ensemble does, however, reliably reduce variance in IC across sources, with IC Std values that are comparable to or better than individual trials.

### 5.3 Per-Sector Patterns

Several consistent patterns emerge across sources:

- **XLK (Technology)** consistently shows the **highest MSE and RMSE** across all sources and variants, reflecting higher daily return volatility in this sector. Percentage errors (MAPE) are also elevated, particularly under sentiment-based features.
- **XLP (Consumer Staples)** and **XLV (Health Care)** tend to show the **lowest MSE/RMSE**, consistent with their defensive, lower-volatility characteristics.
- **XLV is a notable outlier on MAPE** in several configurations — particularly NEWS rank1 (2.36%) and REDDIT rank0 (3.77%) — suggesting that health care sector returns are harder to predict in percentage terms when sentiment signals are noisy or misdirected.
- **XLE (Energy)** shows elevated errors across all sources, which is consistent with energy sector sensitivity to commodity price shocks that may not be well-captured by equity-focused technical or sentiment features.

### 5.4 IC Analysis

IC values are uniformly **low but positive on average** (IC Mean in the range 0.009 to 0.026 for ensembles), with **high standard deviation** (~0.31–0.37). This is expected in financial forecasting: any predictive signal tends to be weak and noisy at the daily frequency. Importantly, positive IC means the model's cross-sectional ranking is marginally better than random, which is sufficient to generate portfolio value over many time steps.

The **negative IC mean observed for rank0 (NEWS)** (IC = −0.004) and **rank1 (REDDIT)** (IC = −0.010) indicates that individual top trials can overfit to validation and reverse in test — a known issue in financial ML. The ensemble mitigates but does not eliminate this risk.

---

## 6. Key Takeaways

1. **Technical features (TECH) produce the most reliable standalone signal**, with the best balance of low MAPE and positive IC for the ensemble. This is a strong baseline and suggests that price-derived features remain the primary driver in this daily-frequency setting.

2. **Combining Reddit and news sentiment (REDDIT+NEWS) improves cross-sectional rank ordering** (highest Rank IC), even though absolute errors are not the best. This is the combination most relevant for Black–Litterman portfolio construction, where ranking assets matters more than point predictions.

3. **The ensemble consistently outperforms individual trials on IC stability**, justifying the top-N approach. However, care is needed when lower-ranked trials from noisy sources (e.g., Reddit alone) drag down ensemble quality.

4. **High-volatility sectors (XLK, XLE) are harder to predict** in absolute terms and contribute disproportionately to aggregate RMSE. Sector-specific error weighting in the loss or BL confidence parameter could help manage this.

5. **IC values, while low, are positive and consistent**, providing sufficient directional signal for portfolio construction, particularly when aggregated over many daily time steps. This aligns with the broader literature on weak but persistent factors in equity markets.

---

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
all_files = sorted([f for f in os.listdir() if f.endswith('.csv')])
print(all_files)

['news_lstm_ensemble_real_vs_pred.csv', 'news_lstm_rank0_real_vs_pred.csv', 'news_lstm_rank1_real_vs_pred.csv', 'news_lstm_rank2_real_vs_pred.csv', 'reddit+news_lstm_ensemble_real_vs_pred.csv', 'reddit+news_lstm_rank0_real_vs_pred.csv', 'reddit+news_lstm_rank1_real_vs_pred.csv', 'reddit+news_lstm_rank2_real_vs_pred.csv', 'reddit_lstm_ensemble_real_vs_pred.csv', 'reddit_lstm_rank0_real_vs_pred.csv', 'reddit_lstm_rank1_real_vs_pred.csv', 'reddit_lstm_rank2_real_vs_pred.csv', 'tech_lstm_ensemble_real_vs_pred.csv', 'tech_lstm_rank0_real_vs_pred.csv', 'tech_lstm_rank1_real_vs_pred.csv', 'tech_lstm_rank2_real_vs_pred.csv']


In [3]:
def compute_metrics(df):
    sectors = sorted(set(col.split('_')[0] for col in df.columns if '_real' in col))

    results = []
    for sector in sectors:
        real = df[f"{sector}_real"]
        pred = df[f"{sector}_pred"]
        mask = real.notna() & pred.notna()
        real, pred = real[mask], pred[mask]

        mse  = np.mean((real - pred) ** 2)
        rmse = np.sqrt(mse)
        non_zero = real != 0
        mape = np.mean(np.abs((real[non_zero] - pred[non_zero]) / real[non_zero]))
        results.append({"Sector": sector, "MSE": mse, "RMSE": rmse, "MAPE": mape})

    metrics_df = pd.DataFrame(results)

    real_mat = pd.DataFrame({s: df[f"{s}_real"] for s in sectors})
    pred_mat = pd.DataFrame({s: df[f"{s}_pred"] for s in sectors})

    ic_list, rank_ic_list = [], []
    for t in range(len(df)):
        real_row = real_mat.iloc[t]
        pred_row = pred_mat.iloc[t]
        mask = real_row.notna() & pred_row.notna()
        if mask.sum() < 2:
            continue
        ic_list.append(np.corrcoef(real_row[mask], pred_row[mask])[0, 1])
        rank_ic_list.append(
            pd.Series(real_row[mask]).corr(pd.Series(pred_row[mask]), method='spearman')
        )

    return metrics_df, pd.Series(ic_list), pd.Series(rank_ic_list)


def format_block(label, metrics_df, ic_series, rank_ic_series):
    mean_vals = metrics_df[["MSE", "RMSE", "MAPE"]].mean()
    std_vals  = metrics_df[["MSE", "RMSE", "MAPE"]].std()

    lines = [
        "=" * 60,
        f"  Variant: {label}",
        "=" * 60,
        "",
        "--- Per-Sector Error Metrics ---",
        metrics_df.to_string(index=False),
        "",
        "--- Aggregate (across sectors) ---",
        f"{'Metric':<8}  {'Mean':>12}  {'Std':>12}",
        "-" * 36,
    ]
    for col in ["MSE", "RMSE", "MAPE"]:
        lines.append(f"{col:<8}  {mean_vals[col]:>12.6f}  {std_vals[col]:>12.6f}")
    lines += [
        "",
        "--- IC Metrics (cross-sector, per time step) ---",
        f"IC Mean:      {ic_series.mean():.6f}",
        f"IC Std:       {ic_series.std():.6f}",
        f"Rank IC Mean: {rank_ic_series.mean():.6f}",
        f"Rank IC Std:  {rank_ic_series.std():.6f}",
    ]
    return "\n".join(lines)


# 'reddit+news' listed before 'reddit' to avoid prefix collision
SOURCES       = ["reddit+news", "news", "reddit", "tech"]
VARIANT_ORDER = ["ensemble", "rank0", "rank1", "rank2"]

grouped = {src: [] for src in SOURCES}
for f in all_files:
    for src in SOURCES:
        if f.startswith(src + "_"):
            grouped[src].append(f)
            break

def variant_key(fname):
    for i, v in enumerate(VARIANT_ORDER):
        if v in fname:
            return i
    return 99

for src in SOURCES:
    files = sorted(grouped[src], key=variant_key)
    if not files:
        print(f"No files found for '{src}', skipping.")
        continue

    blocks = [
        "#" * 60,
        f"  Source: {src.upper()}",
        "#" * 60,
        "",
    ]

    for fname in files:
        df    = pd.read_csv(fname)
        label = fname.replace(src + "_lstm_", "").replace("_real_vs_pred.csv", "")
        metrics_df, ic_series, rank_ic_series = compute_metrics(df)
        blocks.append(format_block(label, metrics_df, ic_series, rank_ic_series))
        blocks.append("")

    out_path = f"metrics_{src}.txt"
    with open(out_path, "w") as fh:
        fh.write("\n".join(blocks))

    print(f"Saved  {out_path}  ({len(files)} variants)")

Saved  metrics_reddit+news.txt  (4 variants)
Saved  metrics_news.txt  (4 variants)
Saved  metrics_reddit.txt  (4 variants)
Saved  metrics_tech.txt  (4 variants)
